# Background to signal CP tags
## Calculate background to signal ratios for CP tags

### Include library for handling uncertainties
#### [Here is the ```uncertainties-cpp``` library on GitHub](https://github.com/Gattocrucco/uncertainties-cpp)

In [1]:
gInterpreter->AddIncludePath("/data/lhcb/users/tat/uncertainties-cpp");

In [2]:
#include<uncertainties/impl.hpp>
#include<uncertainties/ureal.hpp>
#include<uncertainties/io.hpp>
#include<uncertainties/math.hpp>
#include<uncertainties/stat.hpp>

### Load utility functions

In [3]:
gROOT->ProcessLine(".L ../UtilityFunctions.C");

### Number of bins

In [4]:
const int NumberBins = 4;

### Get reconstructed background bin yields

In [5]:
std::map<int, double> GetRecBackgroundBinYields(const std::string &SignalMode,
                                                std::string BackgroundMode) {
    if(BackgroundMode == "KSpi0_KS2pi0pi0") {
        BackgroundMode = "KSpi0";
    }
    std::string Filename = "${BES3_ANALYSIS_PATH}/Selection/PeakingBackgrounds/DoubleTag/";
    Filename += SignalMode + "/KKpipi_vs_" + BackgroundMode + "_to_KKpipi_vs_";
    Filename += SignalMode + "_DoubleTag_SignalMC_Binned.root";
    TChain Chain((SignalMode + "DoubleTag").c_str());
    Chain.Add(Filename.c_str());
    return GetBinYields(&Chain, true, NumberBins);
}

### Get generated bin yields

In [6]:
std::map<int, double> GetGenBinYields(std::string TagMode) {
    if(TagMode.find("PartReco") != std::string::npos) {
        TagMode = TagMode.substr(0, TagMode.length() - 8);
    }
    if(TagMode == "KSpi0_KS2pi0pi0") {
        TagMode = "KSpi0";
    }
    std::string Filename = "${BES3_ANALYSIS_PATH}/TruthTuples/BinnedTruthTuples/";
    Filename += TagMode + "/KKpipi_vs_" + TagMode + "_TruthTuple_Binned.root";
    TChain Chain("TruthTuple");
    Chain.Add(Filename.c_str());
    return GetBinYields(&Chain, false, NumberBins);
}

### List of tags and their backgrounds

In [7]:
const std::map<std::string, std::vector<std::string>> Tags{
    {"pipipi0", std::vector<std::string>{"KSpi0"}},
    {"KSpi0", std::vector<std::string>{"pipipi0"}},
    {"KSeta", std::vector<std::string>{"pipieta"}},
    {"KSetaPrimerhogamma", std::vector<std::string>{"KSpipipi0"}},
    {"KSpi0pi0", std::vector<std::string>{"pipipi0pi0", "KSKS", "KSpi0gamma"}},
    {"KLpi0", std::vector<std::string>{"KLpi0pi0", "pi0pi0", "pi0eta", "KSpi0_KS2pi0pi0"}},
    {"pipipi0PartReco", std::vector<std::string>{"KSpi0"}},
    {"KSpi0PartReco", std::vector<std::string>{"pipipi0"}}
};

### Start calculating the background to signal bin efficiencies, times the ratio of branching fractions

In [8]:
std::string BackgroundToSignalRatios;
for(const auto &Tag : Tags) {
    for(std::size_t i = 0; i < Tag.second.size(); i++) {
        const auto SignalBF = GetBranchingFraction(Tag.first);
        uncertainties::udouble SignalBF_unc(SignalBF.first, SignalBF.second);
        const auto BackgroundBF = GetBranchingFraction(Tag.second[i]);
        uncertainties::udouble BackgroundBF_unc(BackgroundBF.first, BackgroundBF.second);
        const auto BFRatio = BackgroundBF_unc/SignalBF_unc;
        const auto SignalRecYields = GetRecSignalBinYields(Tag.first, NumberBins);
        const auto SignalGenYields = GetGenBinYields(Tag.first);
        const auto BackgroundRecYields = GetRecBackgroundBinYields(Tag.first, Tag.second[i]);
        const auto BackgroundGenYields = GetGenBinYields(Tag.second[i]);
        std::vector<uncertainties::udouble> BkgToSigRatio;
        for(int Bin = 1; Bin <= NumberBins; Bin++) {
            std::string Label = Tag.first + "_PeakingBackground" + std::to_string(i + 1);
            Label += "_DoubleTag_CP_KKpipi_vs_" + Tag.first + "_SignalBin";
            Label += std::to_string(Bin) + "_BackgroundToSignalRatio";
            const double SigEff = SignalRecYields.at(Bin)/SignalGenYields.at(Bin);
            const double SigEff_err = TMath::Sqrt(SigEff*(1.0 - SigEff)/SignalGenYields.at(Bin));
            const uncertainties::udouble SigEff_unc(SigEff, SigEff_err);
            const double BkgEff = BackgroundRecYields.at(Bin)/BackgroundGenYields.at(Bin);
            const double BkgEff_err = TMath::Sqrt(BkgEff*(1.0 - BkgEff)/BackgroundGenYields.at(Bin));
            const uncertainties::udouble BkgEff_unc(BkgEff, BkgEff_err);
            const auto EffRatio = BkgEff_unc/SigEff_unc;
            BkgToSigRatio.push_back(EffRatio*BFRatio);
            BackgroundToSignalRatios += Label + " ";
            BackgroundToSignalRatios += std::to_string(uncertainties::nom(BkgToSigRatio.back())) + "\n";
            BackgroundToSignalRatios += Label + "_err ";
            BackgroundToSignalRatios += std::to_string(uncertainties::sdev(BkgToSigRatio.back())) + "\n";
        }
        BackgroundToSignalRatios += "\n";
        std::string Filename = "PeakingBackground_DT_" + Tag.second[i] + "_to_" + Tag.first + ".root";
        std::vector<double> FlatCovMatrix =
            uncertainties::cov_matrix<std::vector<double>>(BkgToSigRatio);
        SaveCovMatrix(FlatCovMatrix, Filename);
    }
}
//std::cout << BackgroundToSignalRatios;

### Save parameters to a file

In [9]:
std::ofstream File("BackgroundToSignalRatios_CP.txt");
File << BackgroundToSignalRatios;
File.close();